In [ ]:
# finance_chatbot.py
import os
import re
import tempfile
from typing import List, Dict, Any
from dotenv import load_dotenv
from googletrans import Translator

# Load environment variables
load_dotenv()

# Import LangChain components
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings, HuggingFacePipeline
from langchain.prompts import ChatPromptTemplate
from langchain.schema import BaseRetriever
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema.output_parser import StrOutputParser
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch

class FinanceChatbot:
    def __init__(self):
        # Initialize components
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        
        # Initialize the language model
        self.llm = self.setup_llm()
        
        self.vectorstore = None
        self.retriever = None
        self.translator = Translator()
        
        # Set up general finance knowledge
        self.setup_general_finance_knowledge()
        
    def setup_llm(self):
        """Initialize the open-source language model"""
        try:
            # Try to use a smaller model first for better compatibility
            model_name = "microsoft/DialoGPT-medium"
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            model = AutoModelForCausalLM.from_pretrained(model_name)
            
            # Create text generation pipeline
            pipe = pipeline(
                "text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=100,
                do_sample=True,
                temperature=0.7,
                top_p=0.95,
                pad_token_id=tokenizer.eos_token_id
            )
            
            return HuggingFacePipeline(pipeline=pipe)
        except Exception as e:
            print(f"Error loading model: {e}")
            # Fallback to a simpler approach if model loading fails
            return self.create_fallback_llm()
    
    def create_fallback_llm(self):
        """Create a simple fallback LLM for basic responses"""
        from langchain.schema import BaseLanguageModel
        from langchain.schema import HumanMessage, AIMessage
        
        class SimpleFinanceLLM(BaseLanguageModel):
            def _generate(self, messages, stop=None, run_manager=None, **kwargs):
                prompt = messages[0].content if isinstance(messages[0], HumanMessage) else messages[0]
                
                # Simple pattern matching for finance questions
                if "stock" in prompt.lower():
                    response = "Stocks represent ownership in a company. Their value can fluctuate based on market conditions."
                elif "bond" in prompt.lower():
                    response = "Bonds are debt securities where investors loan money to entities for a fixed period at a fixed interest rate."
                elif "mutual fund" in prompt.lower():
                    response = "Mutual funds pool money from many investors to purchase a diversified portfolio of stocks, bonds, or other securities."
                elif "etf" in prompt.lower():
                    response = "ETFs (Exchange-Traded Funds) are investment funds traded on stock exchanges, similar to stocks."
                elif "retirement" in prompt.lower() or "401k" in prompt.lower():
                    response = "Retirement planning involves setting income goals for retirement and taking steps to achieve those goals."
                else:
                    response = "I'm a finance assistant. I can help with questions about stocks, bonds, mutual funds, ETFs, and retirement planning."
                
                return AIMessage(content=response)
            
            async def _agenerate(self, messages, stop=None, run_manager=None, **kwargs):
                return self._generate(messages, stop, run_manager, **kwargs)
            
            @property
            def _llm_type(self):
                return "simple_finance_llm"
        
        return SimpleFinanceLLM()
        
    def setup_general_finance_knowledge(self):
        """Initialize with some general finance knowledge"""
        general_finance_docs = [
            "A stock represents ownership in a company and constitutes a claim on part of the company's assets and earnings.",
            "A bond is a fixed income instrument that represents a loan made by an investor to a borrower.",
            "Diversification is a risk management strategy that mixes a wide variety of investments within a portfolio.",
            "Compound interest is the interest on a loan or deposit calculated based on both the initial principal and the accumulated interest from previous periods.",
            "A mutual fund is a professionally managed investment fund that pools money from many investors to purchase securities.",
            "The stock market refers to public markets that exist for issuing, buying, and selling stocks that trade on a stock exchange or over-the-counter.",
            "An ETF (Exchange-Traded Fund) is a type of security that tracks an index, sector, commodity, or other asset but can be purchased or sold on a stock exchange.",
            "A 401(k) is a retirement savings plan sponsored by an employer that lets workers save and invest a portion of their paycheck before taxes are taken out.",
            "Inflation is the rate at which the general level of prices for goods and services is rising, and subsequently, purchasing power is falling.",
            "A bull market is a period of rising stock prices, while a bear market is a period of falling stock prices."
        ]
        
        # Create a small vector store with general finance knowledge
        self.vectorstore = FAISS.from_texts(
            general_finance_docs, 
            self.embeddings
        )
        self.retriever = self.vectorstore.as_retriever()
        
    def process_annual_report(self, file_path: str):
        """Process an uploaded annual report PDF"""
        try:
            # Load the PDF
            loader = PyPDFLoader(file_path)
            documents = loader.load()
            
            # Split the document into chunks
            text_splitter = RecursiveCharacterTextSplitter(
                chunk_size=1000, 
                chunk_overlap=200
            )
            splits = text_splitter.split_documents(documents)
            
            # Add to vector store
            if self.vectorstore is None:
                self.vectorstore = FAISS.from_documents(
                    documents=splits, 
                    embedding=self.embeddings
                )
            else:
                # Add to existing vector store
                self.vectorstore.add_documents(splits)
                
            self.retriever = self.vectorstore.as_retriever()
            return True, "Annual report processed successfully!"
            
        except Exception as e:
            return False, f"Error processing annual report: {str(e)}"
    
    def detect_language(self, text: str) -> str:
        """Detect the language of the input text"""
        try:
            detection = self.translator.detect(text)
            return detection.lang
        except:
            return "en"  # Default to English if detection fails
    
    def translate_text(self, text: str, target_lang: str = "en") -> str:
        """Translate text to the target language"""
        if target_lang == "en":
            return text
            
        try:
            translation = self.translator.translate(text, dest=target_lang)
            return translation.text
        except:
            return text  # Return original text if translation fails
    
    def format_docs(self, docs):
        """Format documents for the prompt"""
        return "\n\n".join(doc.page_content for doc in docs)
    
    def generate_response(self, query: str, language: str = "en") -> str:
        """Generate a response to the user query"""
        # Translate non-English queries to English for processing
        if language != "en":
            query_en = self.translate_text(query, "en")
        else:
            query_en = query
        
        # Set up the RAG chain
        template = """You are a helpful finance assistant. Use the following context to answer the question.
        If you don't know the answer, just say that you don't know. Don't try to make up an answer.
        Use three sentences maximum and keep the answer concise.
        
        Context: {context}
        
        Question: {question}
        
        Answer:"""
        prompt = ChatPromptTemplate.from_template(template)
        
        rag_chain = (
            {"context": self.retriever | self.format_docs, "question": RunnablePassthrough()}
            | prompt
            | self.llm
            | StrOutputParser()
        )
        
        # Generate response
        try:
            response_en = rag_chain.invoke(query_en)
        except Exception as e:
            print(f"Error generating response: {e}")
            response_en = "I'm sorry, I encountered an error while processing your request. Please try again."
        
        # Translate response back to original language if needed
        if language != "en":
            response = self.translate_text(response_en, language)
        else:
            response = response_en
            
        return response
    
    def chat(self, query: str) -> str:
        """Main chat method that handles multilingual queries"""
        # Detect language
        language = self.detect_language(query)
        
        # Generate response
        response = self.generate_response(query, language)
        
        return response

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
